# DuckDB Warehouse Exploration

This notebook provides a detailed view of the Sentinel Ecosystem's data warehouse, including table schemas, row counts, and sample data across the Bronze, Silver, and Gold layers.

In [1]:
import duckdb

# Using absolute path for consistency in VS Code
db_path = 'data/sentinel_warehouse.db'
conn = duckdb.connect(db_path, read_only=True)
print(f"Connected to: {db_path}")

Connected to: data/sentinel_warehouse.db


## 1. List All Tables

In [2]:
conn.execute("SHOW TABLES").df()

,name
0,bronze_users
1,gold_user_metrics
2,silver_users


## 2. Table Schemas

In [ ]:
print("Silver Layer Schema:")
conn.execute("DESCRIBE silver_users").df()[['column_name', 'column_type', 'null']]

Silver Layer Schema:


,column_name,column_type,null
0,user_id,VARCHAR,YES
1,name,VARCHAR,YES
2,email,VARCHAR,YES
3,signup_timestamp,TIMESTAMP,YES
4,country,VARCHAR,YES
5,age,BIGINT,YES
6,is_active,BOOLEAN,YES


## 3. Data Volumes (Row Counts)

In [3]:
counts_query = """
SELECT 'bronze_users' as table_name, count(*) as row_count FROM bronze_users
UNION ALL
SELECT 'silver_users' as table_name, count(*) as row_count FROM silver_users
UNION ALL
SELECT 'gold_user_metrics' as table_name, count(*) as row_count FROM gold_user_metrics
"""
conn.execute(counts_query).df()

,table_name,row_count
0,bronze_users,200
1,silver_users,200
2,gold_user_metrics,144


## 4. Sample Data Samples

In [ ]:
print("Gold Layer Sample (Aggregated Metrics):")
display(conn.execute("SELECT * FROM gold_user_metrics LIMIT 5").df())

print("\nSilver Layer Sample (Cleaned Data):")
display(conn.execute("SELECT * FROM silver_users LIMIT 5").df())

Gold Layer Sample (Aggregated Metrics):


,country,total_users,avg_age,active_users
0,Armenia,1,48.0,0
1,Cambodia,2,51.0,2
2,Namibia,2,49.5,1
3,Puerto Rico,1,63.0,1
4,Gibraltar,2,66.0,2



Silver Layer Sample (Cleaned Data):


,user_id,name,email,signup_timestamp,country,age,is_active
0,8ad0c581-b305-4d4d-aa1c-ca9516c5431e,Ashley Myers,tsnow@example.net,2025-05-01 22:01:10.013961,Comoros,49,True
1,58b732a6-807a-4c69-a88a-f63c79161b09,Virginia Sawyer,garcianatalie@example.com,2026-01-05 04:12:52.488241,Equatorial Guinea,31,True
2,80ad5801-98ff-439d-a7cc-2320875cad35,Felicia May,john12@example.com,2025-10-25 09:50:52.492135,Kenya,72,True
3,d497e29f-3b83-40cc-9cbe-8869b8b65a01,Brian Fernandez,shannonberry@example.net,2026-02-25 16:42:47.381464,Mexico,50,True
4,67ac54a7-c538-4b14-9b0d-6c2a47cd61be,Brittany Lara,gillemily@example.org,2025-08-03 15:39:53.899199,Guinea,78,True


## 5. Data Quality Monitoring

Check for NULL values in the Bronze layer, which can indicate issues in upstream data generation or ingestion.

In [4]:
null_check_query = "SELECT COUNT(*) as null_user_id_count FROM bronze_users WHERE user_id IS NULL"
conn.execute(null_check_query).df()

,null_user_id_count
0,20


In [6]:
conn.execute("SELECT * FROM bronze_users WHERE user_id IS NULL limit 5").df()

,user_id,name,email,signup_date,country,age,is_active
0,None,Suzanne Goodwin,chungelizabeth@example.com,2025-09-24 13:59:18.808009,Palau,28,True
1,None,Megan Lee,jonathan16@example.com,2025-07-19 19:24:16.948030,Cambodia,30,True
2,None,Vanessa Phelps,abigailmeyer@example.com,2025-06-22 00:27:05.137603,Anguilla,68,True
3,None,Daniel Gibson,bperez@example.com,2025-08-27 17:03:41.414167,Uganda,56,True
4,None,Brenda Welch,debbie35@example.org,2025-05-31 23:15:50.569052,Congo,72,False
